# Activity 3: Pandas Data Cleaning

Clean a messy stock dataset and prove each cleaning step with a quick check. Run one completed solution cell at a time and review the result.

Solutions notebook for instructor reference. Keep the student drill notebook unchanged.

## Instructor Teaching Note

Use this notebook to normalize the idea that pandas usually has several valid routes. For each step, ask: are we making a Series, adding a column, filtering rows, or building a summary DataFrame?

In [ ]:
import pandas as pd

DATA_PATH = "../Activity_3_Pandas_Data_Cleaning/Resources/stock_data.csv"

## 1. Read and inspect

Load the CSV into `stocks_df`, then inspect the first rows, shape, and column dtypes.

In [ ]:
stocks_df = pd.read_csv(DATA_PATH)
stocks_df.head()
print(stocks_df.shape)
stocks_df.dtypes

## 2. Profile missing values

Calculate the missing count and missing percent for each column.

In [ ]:
missing_counts = stocks_df.isna().sum()
missing_percent = stocks_df.isna().mean() * 100

# Option 1: make a DataFrame from a dictionary of Series.
# This is the clearest option for most beginners.
missing_profile = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent,
})
missing_profile

# Option 2: concat the two Series side by side.
missing_profile_concat = pd.concat([missing_counts, missing_percent], axis=1)
missing_profile_concat.columns = ["missing_count", "missing_percent"]

# Option 3: start with one Series and add the second as a new column.
missing_profile_step_by_step = missing_counts.to_frame(name="missing_count")
missing_profile_step_by_step["missing_percent"] = missing_percent

missing_profile

## 3. Clean money columns

`price` and `earnings_per_share` contain dollar signs. Create numeric versions named `price_numeric` and `eps_numeric`.

In [ ]:
stocks_df["price_numeric"] = pd.to_numeric(
    stocks_df["price"].str.replace("$", "", regex=False),
    errors="coerce",
)
stocks_df["eps_numeric"] = pd.to_numeric(
    stocks_df["earnings_per_share"].str.replace("$", "", regex=False),
    errors="coerce",
)

stocks_df[["price", "price_numeric", "earnings_per_share", "eps_numeric"]].head()

## 4. Handle missing EBITDA

Create `clean_df` from `stocks_df`. Fill missing `ebitda` with 0, then drop rows still missing a critical field: `symbol`, `sector`, `price_numeric`, or `eps_numeric`.

In [ ]:
clean_df = stocks_df.copy()
clean_df["ebitda"] = clean_df["ebitda"].fillna(0)
clean_df = clean_df.dropna(subset=["symbol", "sector", "price_numeric", "eps_numeric"])

print(len(stocks_df))
print(len(clean_df))

## 5. Summarize by sector

Group by `sector` and calculate average price, median price, average EPS, and company count.

In [ ]:
sector_summary = clean_df.groupby("sector").agg({
    "price_numeric": ["mean", "median"],
    "eps_numeric": "mean",
    "symbol": "count",
})
sector_summary

## Reflection

Answer in the markdown cell below: which cleaning step changed the row count, and which step only changed values?

Your answer here.